In [ ]:
import os
import csv
import time
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Cấu hình
SEARCH_KEYWORD = "food"
SCROLL_LIMIT = 10
DOWNLOAD_FOLDER = r'D:\project\Food-Captioning\pexels\food_images'
CSV_FILENAME = 'data.csv'

# Tạo folder
os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)

# Tạo file CSV
with open(CSV_FILENAME, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["Image URL", "Description"])

# Khởi tạo trình duyệt
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)
wait = WebDriverWait(driver, 10)

# Truy cập trang
url = f"https://www.pexels.com/search/{SEARCH_KEYWORD}/"
driver.get(url)

seen_urls = set()
image_count = 0

with open(CSV_FILENAME, mode='a', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    for i in range(SCROLL_LIMIT):
        print(f"🔄 Đang xử lý lượt cuộn {i + 1}/{SCROLL_LIMIT}")
        wait.until(EC.presence_of_all_elements_located((By.TAG_NAME, 'img')))
        photos = driver.find_elements(By.TAG_NAME, 'img')

        for img in photos:
            src = img.get_attribute('src')
            alt = img.get_attribute('alt') or 'No description'
            if src and 'images.pexels.com' in src and src not in seen_urls:
                seen_urls.add(src)
                filename = f"image_{image_count}.jpg"
                filepath = os.path.join(DOWNLOAD_FOLDER, filename)
                try:
                    response = requests.get(src, timeout=10)
                    with open(filepath, 'wb') as img_file:
                        img_file.write(response.content)
                    print(f"📸 Lưu ảnh: {filename}")
                    writer.writerow([src, alt])
                    f.flush()
                    image_count += 1
                except Exception as e:
                    print(f"❌ Lỗi tải ảnh {src}: {e}")

        try:
            load_more = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[contains(text(),"Load more")]')))
            load_more.click()
            time.sleep(2)
        except:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

driver.quit()
print(f"✅ Đã lưu tổng cộng {image_count} ảnh và mô tả vào {CSV_FILENAME}")

🔄 Đang xử lý lượt cuộn 1/10
📸 Lưu ảnh: image_0.jpg
📸 Lưu ảnh: image_1.jpg
📸 Lưu ảnh: image_2.jpg
📸 Lưu ảnh: image_3.jpg
📸 Lưu ảnh: image_4.jpg
📸 Lưu ảnh: image_5.jpg
📸 Lưu ảnh: image_6.jpg
📸 Lưu ảnh: image_7.jpg
📸 Lưu ảnh: image_8.jpg
📸 Lưu ảnh: image_9.jpg
📸 Lưu ảnh: image_10.jpg
📸 Lưu ảnh: image_11.jpg
📸 Lưu ảnh: image_12.jpg
📸 Lưu ảnh: image_13.jpg
📸 Lưu ảnh: image_14.jpg
📸 Lưu ảnh: image_15.jpg
📸 Lưu ảnh: image_16.jpg
📸 Lưu ảnh: image_17.jpg
📸 Lưu ảnh: image_18.jpg
📸 Lưu ảnh: image_19.jpg
📸 Lưu ảnh: image_20.jpg
📸 Lưu ảnh: image_21.jpg
📸 Lưu ảnh: image_22.jpg
📸 Lưu ảnh: image_23.jpg
📸 Lưu ảnh: image_24.jpg
📸 Lưu ảnh: image_25.jpg
📸 Lưu ảnh: image_26.jpg
🔄 Đang xử lý lượt cuộn 2/10
📸 Lưu ảnh: image_27.jpg
📸 Lưu ảnh: image_28.jpg
📸 Lưu ảnh: image_29.jpg
📸 Lưu ảnh: image_30.jpg
📸 Lưu ảnh: image_31.jpg
📸 Lưu ảnh: image_32.jpg
📸 Lưu ảnh: image_33.jpg
📸 Lưu ảnh: image_34.jpg
📸 Lưu ảnh: image_35.jpg
📸 Lưu ảnh: image_36.jpg
📸 Lưu ảnh: image_37.jpg
📸 Lưu ảnh: image_38.jpg
📸 Lưu ảnh: image_3

In [ ]:
import os
import csv
import re
import shutil

# Cấu hình
INPUT_CSV = 'data.csv'
IMAGE_FOLDER = r'D:\project\Food-Captioning\pexels\images'
OUTPUT_CSV = 'processed.csv'

# Hàm trích xuất tên file từ URL
def get_filename_from_url(url):
    # Lấy phần cuối của URL (sau dấu / cuối cùng)
    filename = url.split('/')[-1].split('?')[0]
    # Đảm bảo tên file hợp lệ (chỉ giữ chữ, số, dấu gạch ngang, dấu chấm)
    filename = re.sub(r'[^a-zA-Z0-9\-\.]', '_', filename)
    # Nếu không có đuôi, thêm .jpg
    if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        filename += '.jpg'
    return filename

# Đọc food_images.csv và ánh xạ tên file hiện tại
url_to_filename = {}
with open(INPUT_CSV, mode='r', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)
    for i, row in enumerate(reader):
        # Giả sử tên file hiện tại là image_{index}.jpg hoặc .jpeg
        current_filename = f"image_{i}.jpg"
        if not os.path.exists(os.path.join(IMAGE_FOLDER, current_filename)):
            current_filename = f"image_{i}.jpeg"  # Thử .jpeg nếu .jpg không tồn tại
        url_to_filename[row['Image URL']] = current_filename

# Đổi tên file
rename_mapping = {}  # Lưu ánh xạ tên cũ -> tên mới
for url, current_filename in url_to_filename.items():
    current_filepath = os.path.join(IMAGE_FOLDER, current_filename)
    if not os.path.exists(current_filepath):
        print(f"⚠️ File không tồn tại: {current_filename}")
        continue

    # Trích xuất tên file gốc từ URL
    new_filename = get_filename_from_url(url)
    new_filepath = os.path.join(IMAGE_FOLDER, new_filename)

    # Tránh trùng lặp
    base, ext = os.path.splitext(new_filename)
    counter = 1
    while os.path.exists(new_filepath):
        new_filename = f"{base}_{counter}{ext}"
        new_filepath = os.path.join(IMAGE_FOLDER, new_filename)
        counter += 1

    # Đổi tên file
    try:
        shutil.move(current_filepath, new_filepath)
        print(f"✅ Đổi tên: {current_filename} -> {new_filename}")
        rename_mapping[current_filename] = new_filename
    except Exception as e:
        print(f"❌ Lỗi khi đổi tên {current_filename}: {e}")

# Tạo processed_images.csv với tên file mới
with open(INPUT_CSV, mode='r', encoding='utf-8') as infile, open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as outfile:
    reader = csv.DictReader(infile)
    writer = csv.DictWriter(outfile, fieldnames=['id', 'src', 'alt'])
    writer.writeheader()
    for i, row in enumerate(reader):
        src = row['Image URL']
        alt = row['Description']
        # Tìm tên file hiện tại
        current_filename = f"image_{i}.jpg"
        if not current_filename in rename_mapping:
            current_filename = f"image_{i}.jpeg"
        # Lấy tên file mới
        new_filename = rename_mapping.get(current_filename)
        if new_filename and os.path.exists(os.path.join(IMAGE_FOLDER, new_filename)):
            writer.writerow({
                'id': new_filename,
                'src': src,
                'alt': alt
            })
        else:
            print(f"⚠️ Bỏ qua: {current_filename} (không tìm thấy file mới)")

print(f"✅ Đã đổi tên file và tạo {OUTPUT_CSV}")

✅ Đổi tên: image_0.jpg -> pexels-photo-1640777.jpeg
✅ Đổi tên: image_1.jpg -> pexels-photo-1099680.jpeg
✅ Đổi tên: image_2.jpg -> pexels-photo-958545.jpeg
✅ Đổi tên: image_3.jpg -> pexels-photo-1640774.jpeg
✅ Đổi tên: image_4.jpg -> pexels-photo-1640773.jpeg
✅ Đổi tên: image_5.jpg -> pexels-photo-349609.jpeg
✅ Đổi tên: image_6.jpg -> pexels-photo-1633525.jpeg
✅ Đổi tên: image_7.jpg -> pexels-photo-1128678.jpeg
✅ Đổi tên: image_8.jpg -> pexels-photo-262959.jpeg
✅ Đổi tên: image_9.jpg -> pexels-photo-1391487.jpeg
✅ Đổi tên: image_10.jpg -> pexels-photo-376464.jpeg
✅ Đổi tên: image_11.jpg -> pexels-photo-1448721.jpeg
✅ Đổi tên: image_12.jpg -> pexels-photo-1667427.jpeg
✅ Đổi tên: image_13.jpg -> pexels-photo-11612324.jpeg
✅ Đổi tên: image_14.jpg -> pexels-photo-842571.jpeg
✅ Đổi tên: image_15.jpg -> pexels-photo-784633.jpeg
✅ Đổi tên: image_16.jpg -> pexels-photo-940302.jpeg
✅ Đổi tên: image_17.jpg -> pexels-photo-1199957.jpeg
✅ Đổi tên: image_18.jpg -> pexels-photo-2097090.jpeg
✅ Đổi tên

In [1]:
import os
import shutil
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input

# --- CONFIG --- #
MODEL_PATH = r'D:\project\Food-Captioning\model\best_resnet_model.h5'
INPUT_FOLDER = r'D:\project\Food-Captioning\pexels\images'
OUTPUT_FOOD = 'sorted/food'
OUTPUT_NONFOOD = 'sorted/nonfood'
CSV_PATH = 'processed.csv'
OUTPUT_CSV = 'data_with_class.csv'
VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png')

os.makedirs(OUTPUT_FOOD, exist_ok=True)
os.makedirs(OUTPUT_NONFOOD, exist_ok=True)

# --- ENVIRONMENT SETUP --- #
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# --- LOAD MODEL --- #
model = load_model(MODEL_PATH)

# --- UTILS --- #
def predict_class(img):
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    prediction = model.predict(img_array, verbose=0)
    label = 0 if prediction[0][0] < 0.5 else 1  # 0 = Food, 1 = Non-food
    confidence = 1 - prediction[0][0] if label == 0 else prediction[0][0]
    return label, confidence

def classify_and_move(row):
    img_id = row.id
    # Ensure file has valid extension
    if not any(img_id.lower().endswith(ext) for ext in VALID_EXTENSIONS):
        img_id = img_id + '.jpg'  # Assume .jpg if no extension
    img_path = os.path.join(INPUT_FOLDER, img_id)
    
    if not os.path.exists(img_path):
        print(f"[❌] File not found: {img_path}")
        return img_id, -1, 0.0
    
    try:
        img = image.load_img(img_path, target_size=(224, 224))
        label, confidence = predict_class(img)
        dest_dir = OUTPUT_FOOD if label == 0 else OUTPUT_NONFOOD
        dest_path = os.path.join(dest_dir, os.path.basename(img_path))
        
        # Avoid overwriting by adding suffix if file exists
        base, ext = os.path.splitext(dest_path)
        counter = 1
        while os.path.exists(dest_path):
            dest_path = f"{base}_{counter}{ext}"
            counter += 1
            
        shutil.copy(img_path, dest_path)
        return img_id, label, confidence
    except Exception as e:
        print(f"[❌] Error processing {img_path}: {e}")
        return img_id, -1, 0.0

# --- MAIN PROCESSING --- #
df = pd.read_csv(CSV_PATH)
df['class'] = -1
df['confidence'] = 0.0  # New column for confidence

print("\n🔍 Running classification on images...\n")
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(classify_and_move, row): row.id for row in df.itertuples(index=False)}
    for future in as_completed(futures):
        img_id, label, confidence = future.result()
        if label != -1:
            df.loc[df['id'] == img_id, 'class'] = label
            df.loc[df['id'] == img_id, 'confidence'] = confidence
        print(f"{img_id} → class: {'Food' if label == 0 else 'Non-food' if label == 1 else 'Error'} ({confidence*100:.2f}%)")

df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Completed! Results saved to: {OUTPUT_CSV}")

c:\Users\ASUS\anaconda3\envs\python10\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(



🔍 Running classification on images...



pexels-photo-1640774.jpeg → class: Food (99.48%)
pexels-photo-1633525.jpeg → class: Food (100.00%)
pexels-photo-349609.jpeg → class: Non-food (91.52%)
pexels-photo-1640777.jpeg → class: Food (100.00%)
pexels-photo-1128678.jpeg → class: Food (99.96%)
pexels-photo-1099680.jpeg → class: Food (100.00%)
pexels-photo-958545.jpeg → class: Food (100.00%)
pexels-photo-1640773.jpeg → class: Food (99.34%)
pexels-photo-262959.jpeg → class: Food (100.00%)
pexels-photo-1448721.jpeg → class: Food (99.94%)
pexels-photo-1391487.jpeg → class: Food (100.00%)
pexels-photo-376464.jpeg → class: Food (100.00%)
pexels-photo-1667427.jpeg → class: Non-food (98.47%)
pexels-photo-842571.jpeg → class: Food (100.00%)
pexels-photo-784633.jpeg → class: Non-food (100.00%)
pexels-photo-11612324.jpeg → class: Food (100.00%)
pexels-photo-940302.jpeg → class: Non-food (99.86%)
pexels-photo-70497.jpeg → class: Food (100.00%)
pexels-photo-699953.jpeg → class: Food (100.00%)
pexels-photo-1640772.jpeg → class: Food (100.00%)
